## [Project Title]

#### Team Members: Carlos Figueredo (carlosfc), James Zhu (jazhu)

### Overview

### Motivation

### Data Sources

For this project, we use two data sources. As of 2025, all the data is public and accessible through the web. 

The first data source is the U.S. Centers for Disease Control and Prevention ([CDC](https://www.cdc.gov/)). Specifically, we use data obtained from the [Natality Information Database](https://wonder.cdc.gov/controller/datarequest/D66). From this database, we extract the birth information for the years 2011 to 2013 for all available counties in the United States. This information is stored in the files `<year>.txt` where `<year>` is replaced by the respective year period. This database provides information at the level of counties, which gives us a high granularity of birth information in the U.S.

The second data source is the U.S. Department of Agriculture ([USDA](https://www.usda.gov/)). Particulary, we use the **2013 Rural-Urban Continuum Codes** available in this [link](https://www.ers.usda.gov/data-products/rural-urban-continuum-codes). All the information is provided by the Economic Research Services Division of the USDA. The information can be downloaded as an Excel file (`.xsl` format), and it is available in the `ruralurbancodes2013.xls` file. This dataset also provides information at the level of counties, and it contains counties' degree of urbanization. Metropolitan counties are categorized by their population size, and nonmetropolitan counties are categorized by their degree of urbanization and adjacency to a metro area.

### Data Description

### Data Manipulation

In [71]:
import pandas as pd
import numpy as np
import os, gzip

births = pd.DataFrame()
for y in range(2011, 2014):
    da = pd.read_csv("%4d.txt" % y, delimiter="\t", dtype={"County Code": object})
    da = da[["County", "County Code", "Births"]]
    da["year"] = y
    # print(f"NaN values for year {y} (sample size={len(da)}):\n{da.isna().sum()}")
    births = pd.concat([births, da])

In [74]:
births = births.rename({'County Code':'FIPS'}, axis=1)
births = births.dropna()
unidentified_mask = births['County'].str.contains('Unidentified')
births = births[~ unidentified_mask]
births

,County,FIPS,Births,year
0,"Baldwin County, AL",01003,2157.0,2011
1,"Calhoun County, AL",01015,1418.0,2011
2,"Etowah County, AL",01055,1173.0,2011
3,"Jefferson County, AL",01073,8916.0,2011
4,"Lee County, AL",01081,1536.0,2011
...,...,...,...,...
565,"Rock County, WI",55105,1955.0,2013
566,"Sheboygan County, WI",55117,1232.0,2013
567,"Washington County, WI",55131,1343.0,2013
568,"Waukesha County, WI",55133,3697.0,2013


In [73]:
rucc = pd.read_excel("ruralurbancodes2013.xls", sheet_name=None) # Requires xlrd. Run pip install xlrd
rucc = rucc["Rural-urban Continuum Code 2013"] # Get first sheet
rucc["FIPS"] = ["%05d" % x for x in rucc.FIPS] # FIPS to object dtype
rucc = rucc.dropna()
rucc.head()

,FIPS,State,County_Name,Population_2010,RUCC_2013,Description
0,01001,AL,Autauga County,54571,2.0,"Metro - Counties in metro areas of 250,000 to ..."
1,01003,AL,Baldwin County,182265,3.0,Metro - Counties in metro areas of fewer than ...
2,01005,AL,Barbour County,27457,6.0,"Nonmetro - Urban population of 2,500 to 19,999..."
3,01007,AL,Bibb County,22915,1.0,Metro - Counties in metro areas of 1 million p...
4,01009,AL,Blount County,57322,1.0,Metro - Counties in metro areas of 1 million p...


### Data Visualization